# AI Silent Mental Health Detector - Complete Analysis
## NLP + LSTM Temporal Modeling for Early Detection

This notebook demonstrates the complete pipeline from text analysis to temporal risk prediction.

**Authors:** Production ML Team  
**Date:** 2024  
**Tech Stack:** Transformers, LSTM, Feature Engineering

## 1. Setup and Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings

warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Import our custom modules
from src.preprocessing import TextPreprocessor
from src.model import EmotionDetector
from src.lstm_model import TemporalMentalHealthModel
from src.pipeline import MentalHealthPipeline

print("✅ All imports successful")

## 2. Load Sample Data

In [ ]:
# Sample texts representing different mental health states
sample_texts = [
    # Positive/Healthy
    "I had such a wonderful day today! Accomplished all my goals and feeling really proud.",
    "Feeling grateful for my friends and family. Life is good right now.",
    "Just finished a great workout. Feeling energized and ready to tackle the day!",
    
    # Moderate concern
    "I'm feeling a bit stressed about work, but I think I can manage it.",
    "Today was tough. Had some challenges but trying to stay positive.",
    "Feeling overwhelmed with everything going on, but I know this will pass.",
    
    # Higher concern
    "I just feel so tired all the time. Nothing seems to bring me joy anymore.",
    "Every day feels the same. I'm struggling to find motivation for anything.",
    "I feel completely alone and hopeless. I don't know what to do anymore.",
    
    # Declining pattern (simulating time series)
    "Things are okay. Had some ups and downs but managing.",
    "Feeling more anxious than usual. Work is really piling up.",
    "I'm really struggling today. Everything feels too difficult.",
    "I can't shake this feeling of emptiness. Nothing helps.",
    "I don't see the point anymore. Just going through the motions.",
]

# Create DataFrame
df = pd.DataFrame({
    'text': sample_texts,
    'entry_id': range(1, len(sample_texts) + 1)
})

print(f"Loaded {len(df)} sample texts")
df.head()

## 3. Initialize Pipeline Components

In [ ]:
# Initialize preprocessor
preprocessor = TextPreprocessor()
print("✅ Preprocessor initialized")

# Initialize emotion detector
print("Loading emotion detection model (this may take a minute)...")
emotion_detector = EmotionDetector()
print("✅ Emotion detector initialized")

# Initialize LSTM model
lstm_model = TemporalMentalHealthModel(sequence_length=5, feature_dim=20)
print("✅ LSTM model initialized")

## 4. Text Preprocessing Analysis

In [ ]:
# Demonstrate preprocessing
sample_text = sample_texts[6]  # A concerning text
print("Original text:")
print(f"'{sample_text}'\n")

cleaned = preprocessor.clean_text(sample_text)
print("Cleaned text:")
print(f"'{cleaned}'\n")

features = preprocessor.extract_features(sample_text)
print("Extracted features:")
for key, value in features.items():
    print(f"  {key}: {value:.4f}")

## 5. Emotion Detection

In [ ]:
# Predict emotions for all samples
emotions = []
confidences = []
all_scores = []

for text in sample_texts:
    emotion, confidence, scores = emotion_detector.predict_emotion(text)
    emotions.append(emotion)
    confidences.append(confidence)
    all_scores.append(scores)

df['emotion'] = emotions
df['confidence'] = confidences

print("Emotion distribution:")
print(df['emotion'].value_counts())

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Emotion distribution
df['emotion'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Emotion Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Emotion')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# Confidence distribution
axes[1].hist(confidences, bins=10, color='coral', edgecolor='black')
axes[1].set_title('Confidence Score Distribution', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Confidence')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

## 6. Feature Engineering

In [ ]:
# Extract features for all texts
all_features = [preprocessor.extract_features(text) for text in sample_texts]
features_df = pd.DataFrame(all_features)

print("Feature summary statistics:")
print(features_df.describe())

# Visualize key features
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Negative word ratio
axes[0, 0].plot(features_df['negative_word_ratio'], marker='o', color='red')
axes[0, 0].set_title('Negative Word Ratio Over Time', fontweight='bold')
axes[0, 0].set_xlabel('Entry ID')
axes[0, 0].set_ylabel('Ratio')
axes[0, 0].grid(True, alpha=0.3)

# Sentiment polarity
axes[0, 1].plot(features_df['sentiment_polarity'], marker='o', color='blue')
axes[0, 1].axhline(y=0, color='black', linestyle='--', alpha=0.5)
axes[0, 1].set_title('Sentiment Polarity Over Time', fontweight='bold')
axes[0, 1].set_xlabel('Entry ID')
axes[0, 1].set_ylabel('Polarity')
axes[0, 1].grid(True, alpha=0.3)

# First person usage
axes[1, 0].plot(features_df['first_person_ratio'], marker='o', color='green')
axes[1, 0].set_title('First Person Pronoun Usage', fontweight='bold')
axes[1, 0].set_xlabel('Entry ID')
axes[1, 0].set_ylabel('Ratio')
axes[1, 0].grid(True, alpha=0.3)

# Feature correlation heatmap
key_features = ['negative_word_ratio', 'sentiment_polarity', 'first_person_ratio', 
                'intensity_ratio', 'sentiment_subjectivity']
corr = features_df[key_features].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=axes[1, 1], 
            cbar_kws={'label': 'Correlation'})
axes[1, 1].set_title('Feature Correlation Matrix', fontweight='bold')

plt.tight_layout()
plt.show()

## 7. Mental Health Score Computation

In [ ]:
# Compute mental health scores using rule-based approach
pipeline = MentalHealthPipeline()

mental_health_scores = []

for i, text in enumerate(sample_texts):
    emotion = emotions[i]
    confidence = confidences[i]
    emotion_scores = all_scores[i]
    features = all_features[i]
    
    score = pipeline.compute_mental_health_score(
        text, emotion, confidence, emotion_scores, features
    )
    mental_health_scores.append(score)

df['mental_health_score'] = mental_health_scores

# Visualize scores
fig, ax = plt.subplots(figsize=(15, 6))

colors = ['green' if s > 0.6 else 'orange' if s > 0.3 else 'red' for s in mental_health_scores]
ax.bar(range(len(mental_health_scores)), mental_health_scores, color=colors, alpha=0.7, edgecolor='black')

# Add threshold lines
ax.axhline(y=0.7, color='green', linestyle='--', label='Healthy Threshold', linewidth=2)
ax.axhline(y=0.4, color='orange', linestyle='--', label='Moderate Risk', linewidth=2)
ax.axhline(y=0.3, color='red', linestyle='--', label='High Risk', linewidth=2)

ax.set_title('Mental Health Scores (Rule-Based)', fontsize=16, fontweight='bold')
ax.set_xlabel('Entry ID', fontsize=12)
ax.set_ylabel('Mental Health Score', fontsize=12)
ax.set_ylim(0, 1)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nScore Statistics:")
print(f"Mean: {np.mean(mental_health_scores):.3f}")
print(f"Std: {np.std(mental_health_scores):.3f}")
print(f"Min: {np.min(mental_health_scores):.3f}")
print(f"Max: {np.max(mental_health_scores):.3f}")

## 8. Generate Sequences for LSTM

In [ ]:
# Create feature vectors for each entry
feature_vectors = []

for i in range(len(sample_texts)):
    fv = pipeline.create_feature_vector(
        all_scores[i],
        all_features[i],
        mental_health_scores[i]
    )
    feature_vectors.append(fv)

feature_vectors = np.array(feature_vectors)
print(f"Feature vectors shape: {feature_vectors.shape}")

# Create sequences (sliding window)
sequence_length = 5
sequences = []

for i in range(len(feature_vectors) - sequence_length + 1):
    seq = feature_vectors[i:i+sequence_length]
    sequences.append(seq)

sequences = np.array(sequences)
print(f"Generated {len(sequences)} sequences of shape {sequences.shape}")

## 9. Train LSTM Model

In [ ]:
# Generate synthetic training data
print("Generating synthetic training data...")
X_train, y_train = lstm_model.generate_synthetic_sequences(n_sequences=1000, decline_ratio=0.35)

print(f"Training data: X={X_train.shape}, y={y_train.shape}")
print(f"Positive samples (decline): {np.sum(y_train)}")
print(f"Negative samples (stable): {len(y_train) - np.sum(y_train)}")

# Build model
lstm_model.build_model()

# Train model
print("\nTraining LSTM model...")
history = lstm_model.train(X_train, y_train, epochs=50, batch_size=32)

## 10. Visualize Training Progress

In [ ]:
# Plot training history
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Loss
axes[0, 0].plot(history.history['loss'], label='Training Loss', linewidth=2)
axes[0, 0].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
axes[0, 0].set_title('Model Loss', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Accuracy
axes[0, 1].plot(history.history['accuracy'], label='Training Accuracy', linewidth=2)
axes[0, 1].plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
axes[0, 1].set_title('Model Accuracy', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# AUC
axes[1, 0].plot(history.history['auc'], label='Training AUC', linewidth=2)
axes[1, 0].plot(history.history['val_auc'], label='Validation AUC', linewidth=2)
axes[1, 0].set_title('Model AUC', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('AUC')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Precision & Recall
axes[1, 1].plot(history.history['precision'], label='Precision', linewidth=2)
axes[1, 1].plot(history.history['recall'], label='Recall', linewidth=2)
axes[1, 1].set_title('Precision & Recall', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Score')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nFinal Training Accuracy: {history.history['accuracy'][-1]:.4f}")
print(f"Final Validation Accuracy: {history.history['val_accuracy'][-1]:.4f}")
print(f"Final AUC: {history.history['val_auc'][-1]:.4f}")

## 11. LSTM Predictions on Sample Sequences

In [ ]:
# Make predictions on our sample sequences
lstm_predictions = []
risk_levels = []

for seq in sequences:
    risk_score, risk_level = lstm_model.predict_risk(seq)
    lstm_predictions.append(risk_score)
    risk_levels.append(risk_level)

# Create results DataFrame
lstm_results = pd.DataFrame({
    'sequence_id': range(1, len(sequences) + 1),
    'lstm_risk_score': lstm_predictions,
    'risk_level': risk_levels,
    'avg_mental_health_score': [seq[:, 0].mean() for seq in sequences]
})

print("LSTM Predictions:")
print(lstm_results)

# Visualize
fig, ax = plt.subplots(figsize=(15, 6))

x = range(len(lstm_predictions))
colors = ['green' if rl == 'low' else 'orange' if rl == 'moderate' else 'red' for rl in risk_levels]

ax.bar(x, lstm_predictions, color=colors, alpha=0.7, edgecolor='black', label='LSTM Risk Score')
ax.plot(x, lstm_results['avg_mental_health_score'], marker='o', color='blue', 
        linewidth=2, markersize=8, label='Avg Mental Health Score (inverted)')

ax.axhline(y=0.3, color='green', linestyle='--', linewidth=2, alpha=0.5)
ax.axhline(y=0.6, color='red', linestyle='--', linewidth=2, alpha=0.5)

ax.set_title('LSTM Risk Predictions vs Rule-Based Scores', fontsize=16, fontweight='bold')
ax.set_xlabel('Sequence ID', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_ylim(0, 1)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 12. Comparison: Rule-Based vs LSTM

In [ ]:
# Compare approaches
print("=" * 80)
print("RULE-BASED vs LSTM COMPARISON")
print("=" * 80)

print("\n📊 RULE-BASED APPROACH:")
print("-" * 80)
print("Pros:")
print("  ✓ Interpretable - clear why a score was assigned")
print("  ✓ Works with single data points")
print("  ✓ Fast computation")
print("  ✓ No training required")
print("\nCons:")
print("  ✗ Cannot detect temporal patterns")
print("  ✗ Misses gradual decline")
print("  ✗ Each entry evaluated independently")
print("  ✗ May miss context from history")

print("\n🧠 LSTM TEMPORAL APPROACH:")
print("-" * 80)
print("Pros:")
print("  ✓ Detects patterns over time")
print("  ✓ Identifies gradual decline")
print("  ✓ Considers historical context")
print("  ✓ More robust to noise")
print("  ✓ Can predict future risk")
print("\nCons:")
print("  ✗ Requires sequence of entries")
print("  ✗ Needs training data")
print("  ✗ Less interpretable")
print("  ✗ Higher computational cost")

print("\n💡 BEST PRACTICE:")
print("-" * 80)
print("Use BOTH approaches in combination:")
print("  1. Rule-based for immediate assessment of each entry")
print("  2. LSTM for temporal pattern detection and risk prediction")
print("  3. Alert when BOTH methods indicate high risk")
print("=" * 80)

## 13. Real-World Application Example

In [ ]:
# Simulate a user's journey over time
user_journey = [
    "Week 1: Feeling good! Started a new project at work and it's going well.",
    "Week 2: Still positive. Had a great weekend with friends.",
    "Week 3: Work is getting more demanding but I'm handling it.",
    "Week 4: Feeling a bit stressed. Project deadlines are approaching.",
    "Week 5: Really overwhelmed. Not sleeping well. Work is consuming everything.",
    "Week 6: I'm exhausted. Nothing seems to help. Feeling hopeless.",
    "Week 7: I don't even want to get out of bed. Everything feels pointless."
]

print("Analyzing user journey over 7 weeks...\n")

# Process each entry
journey_results = []
for i, text in enumerate(user_journey, 1):
    result = pipeline.process_text(text, store_history=True)
    journey_results.append(result)
    
    print(f"Week {i}:")
    print(f"  Emotion: {result['emotion'].upper()} ({result['emotion_confidence']:.1%})")
    print(f"  Mental Health Score: {result['mental_health_score']:.3f}")
    
    if i >= 5:
        lstm_pred = pipeline.lstm_predict()
        if lstm_pred['has_prediction']:
            print(f"  🔮 LSTM Risk: {lstm_pred['risk_level'].upper()} ({lstm_pred['risk_score']:.3f})")
    print()

# Visualize the journey
journey_scores = [r['mental_health_score'] for r in journey_results]

fig, ax = plt.subplots(figsize=(15, 6))

weeks = range(1, len(journey_scores) + 1)
ax.plot(weeks, journey_scores, marker='o', linewidth=3, markersize=10, color='darkblue')

# Color zones
ax.axhspan(0.7, 1.0, alpha=0.2, color='green', label='Healthy Zone')
ax.axhspan(0.4, 0.7, alpha=0.2, color='orange', label='Moderate Risk')
ax.axhspan(0.0, 0.4, alpha=0.2, color='red', label='High Risk')

# Mark LSTM activation point
ax.axvline(x=5, color='purple', linestyle='--', linewidth=2, label='LSTM Activated')

ax.set_title('User Mental Health Journey Over 7 Weeks', fontsize=16, fontweight='bold')
ax.set_xlabel('Week', fontsize=12)
ax.set_ylabel('Mental Health Score', fontsize=12)
ax.set_ylim(0, 1)
ax.set_xticks(weeks)
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n⚠️ CRITICAL ALERT:")
print("This user shows a clear declining pattern from Week 1 to Week 7.")
print("The LSTM model would have detected this trend and flagged for intervention.")
print("Early detection enables timely support and prevention.")

## 14. Save Model

In [ ]:
# Save the trained LSTM model
model_path = "models/mental_health_lstm"
import os
os.makedirs("models", exist_ok=True)

lstm_model.save_model(model_path)
print(f"✅ Model saved to {model_path}")

# To load later:
# lstm_model.load_model(model_path)

## 15. Conclusion & Next Steps

In [ ]:
print("="*80)
print("PROJECT SUMMARY: AI Silent Mental Health Detector")
print("="*80)

print("\n✅ COMPLETED COMPONENTS:")
print("  1. ✓ Text Preprocessing with linguistic feature extraction")
print("  2. ✓ Transformer-based emotion detection (DistilRoBERTa)")
print("  3. ✓ Multi-dimensional feature engineering")
print("  4. ✓ Rule-based mental health scoring")
print("  5. ✓ LSTM temporal pattern detection")
print("  6. ✓ Hybrid prediction system")

print("\n🎯 KEY ACHIEVEMENTS:")
print(f"  • Trained LSTM model with {history.history['val_accuracy'][-1]:.1%} validation accuracy")
print(f"  • Detected declining pattern in user journey example")
print(f"  • Integrated 7 emotion categories + 15+ linguistic features")
print(f"  • Production-ready pipeline with {len(journey_results)} test entries")

print("\n🚀 NEXT STEPS FOR PRODUCTION:")
print("  1. Deploy Streamlit app (run: streamlit run app.py)")
print("  2. Integrate with real user database")
print("  3. Implement automated alerts for high-risk cases")
print("  4. Add professional review dashboard")
print("  5. Fine-tune LSTM on domain-specific data")
print("  6. Add multilingual support")
print("  7. Implement privacy-preserving techniques")
print("  8. Create mobile app interface")

print("\n📊 PERFORMANCE METRICS:")
print(f"  • LSTM Training Accuracy: {history.history['accuracy'][-1]:.4f}")
print(f"  • LSTM Validation Accuracy: {history.history['val_accuracy'][-1]:.4f}")
print(f"  • AUC Score: {history.history['val_auc'][-1]:.4f}")
print(f"  • Precision: {history.history['precision'][-1]:.4f}")
print(f"  • Recall: {history.history['recall'][-1]:.4f}")

print("\n⚠️ IMPORTANT DISCLAIMER:")
print("This is a screening tool, NOT a diagnostic tool.")
print("Always consult mental health professionals for assessment and treatment.")
print("="*80)